In [ ]:
def compare():
    # /// script
    # requires-python = ">=3.10,<3.13"
    # dependencies = [
    #     "transformers", "torch", "torchaudio", "numpy", "pandas",
    #     "scikit-learn", "tqdm", "soundfile", "huggingface_hub"
    # ]
    # ///

    import os
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
    os.environ["USE_TORCHVISION"] = "0"
    os.environ["TRANSFORMERS_NO_TORCHVISION_IMAGE"] = "1"

    import sys, types, enum, random, time, gc, urllib.request, multiprocessing, hashlib
    import importlib, importlib.machinery
    from pathlib import Path
    from collections import OrderedDict

    # ---------------------------------------------------------------------------
    # 0. Dynamic torchvision stub
    # ---------------------------------------------------------------------------
    _BROKEN_TV = True
    try:
        import torchvision  # noqa
        from torchvision.transforms import InterpolationMode  # noqa
        from torchvision.models import resnet50  # noqa
        _BROKEN_TV = False
    except Exception:
        pass

    def _install_stub(name, is_package=True):
        mod = types.ModuleType(name)
        spec = importlib.machinery.ModuleSpec(name=name, loader=None, is_package=is_package)
        if is_package:
            spec.submodule_search_locations = []
        mod.__spec__ = spec
        mod.__loader__ = None
        mod.__package__ = name if is_package else name.rpartition(".")[0]
        mod.__path__ = [] if is_package else None
        mod.__file__ = f"<stub:{name}>"
        mod.__version__ = "0.0.0-stub"

        def _mod_getattr(attr, _name=name):
            if attr.startswith("__") and attr.endswith("__"):
                raise AttributeError(attr)
            cls = type(attr, (), {"__module__": _name})
            setattr(sys.modules[_name], attr, cls)
            return cls

        mod.__getattr__ = _mod_getattr
        sys.modules[name] = mod
        return mod

    if _BROKEN_TV:
        fake = _install_stub("torchvision", True)
        tr = _install_stub("torchvision.transforms", True)
        class InterpolationMode(enum.Enum):
            NEAREST = "nearest"; NEAREST_EXACT = "nearest-exact"; BILINEAR = "bilinear"
            BICUBIC = "bicubic"; BOX = "box"; HAMMING = "hamming"; LANCZOS = "lanczos"
        tr.InterpolationMode = InterpolationMode
        tr.functional = _install_stub("torchvision.transforms.functional", False)
        tr_v2 = _install_stub("torchvision.transforms.v2", True)
        tr_v2.InterpolationMode = InterpolationMode
        tr_v2.functional = _install_stub("torchvision.transforms.v2.functional", False)
        tr.v2 = tr_v2
        fake.transforms = tr
        fake.io = _install_stub("torchvision.io", False)
        fake.io.ImageReadMode = type("ImageReadMode", (), {"UNCHANGED":0,"GRAY":1,"GRAY_ALPHA":2,"RGB":3,"RGBA":4})
        fake.models = _install_stub("torchvision.models", True)
        fake.ops = _install_stub("torchvision.ops", False)
        fake.tv_tensors = _install_stub("torchvision.tv_tensors", False)
        fake.utils = _install_stub("torchvision.utils", False)
        for sub in ("torchvision.datasets", "torchvision.datasets.folder",
                    "torchvision.models.detection", "torchvision.models.segmentation",
                    "torchvision.models.video", "torchvision.models.quantization"):
            _install_stub(sub, True)
        print("[stub] torchvision ok (dynamic)")

    _MP_CTX = None
    if os.name == "posix":
        try:
            _MP_CTX = multiprocessing.get_context("fork")
            print("Using fork multiprocessing context")
        except Exception:
            _MP_CTX = None

    import torch.nn as nn
    _orig_getattr = nn.Module.__getattr__
    nn.Module.__getattr__ = lambda self, name: {} if name == "all_tied_weights_keys" else _orig_getattr(self, name)

    import numpy as np, pandas as pd, torch, torch.nn.functional as F, torchaudio
    from tqdm import tqdm
    from sklearn.metrics import roc_curve

    torch.backends.cudnn.benchmark = False

    def sseed(*p):
        t = "||".join(str(x) for x in p)
        return int.from_bytes(hashlib.sha256(t.encode()).digest()[:8], "little")

    # ---------------------------------------------------------------------------
    # 1. CONFIG
    # ---------------------------------------------------------------------------
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_CUDA = DEVICE.startswith("cuda")
    print(f"Device: {DEVICE}")
    if USE_CUDA:
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")

    TEST_ROOT = "/marimo/voxceleb1_test/wav"
    TRIAL_FILE = "/marimo/veri_test.txt"
    MUSAN_ROOT = "/marimo/musan"

    SR = 16000
    NOISE_SUBDIRS = [("babble", "speech"), ("music", "music"), ("noise", "noise")]
    MUSAN_TEST_RATIO = 0.30
    MUSAN_SPLIT_SEED_BASE = 1234
    EVAL_SNRS = [-20, -10, 0, 10, 20]
    PAPER_SNRS = [0, -10, -20]

    # ---------------------------------------------------------------------------
    # 2. AUDIO UTILITIES
    # ---------------------------------------------------------------------------
    AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3"}
    _AC = OrderedDict()
    def _cp(k, v):
        if k in _AC: _AC.move_to_end(k); return
        _AC[k] = v
        while len(_AC) > 5000: _AC.popitem(last=False)

    def list_audio(root):
        if not os.path.isdir(root): return []
        return sorted(str(p) for p in Path(root).rglob("*")
                      if p.is_file() and p.suffix.lower() in AUDIO_EXTS)

    def load_wav(p, sr_t=SR):
        k = (str(p), sr_t)
        v = _AC.get(k)
        if v is not None: _AC.move_to_end(k); return v
        try: wav, sr = torchaudio.load(str(p), backend="soundfile")
        except Exception: wav, sr = torchaudio.load(str(p))
        if sr != sr_t: wav = torchaudio.functional.resample(wav, sr, sr_t)
        if wav.dim() == 2: wav = wav.mean(dim=0)
        wav = wav.float().contiguous(); _cp(k, wav); return wav

    def rms(x, eps=1e-8): return torch.sqrt(torch.mean(x ** 2) + eps)

    def mix_det(c, n, snr, seed_key):
        r = random.Random(sseed(*seed_key))
        if n.numel() < c.numel(): n = n.repeat(int(np.ceil(c.numel() / n.numel())))
        if n.numel() > c.numel():
            s = r.randint(0, n.numel() - c.numel()); n = n[s:s + c.numel()]
        else:
            n = n[:c.numel()]
        sc = rms(c) / (rms(n) * (10 ** (snr / 20)) + 1e-8)
        nc = sc * n; no = c + nc
        pk = torch.maximum(c.abs().max(), no.abs().max()).clamp_min(1e-6)
        return no / pk, c / pk, nc / pk

    def compute_eer(sc, lb):
        s = np.asarray(sc, dtype=np.float64); l = np.asarray(lb, dtype=np.int64)
        if len(np.unique(l)) < 2: return 50.0
        fpr, tpr, _ = roc_curve(l, s); fnr = 1 - tpr
        i = np.nanargmin(np.abs(fnr - fpr)); return float((fpr[i] + fnr[i]) / 2 * 100)

    def split_strat(files, r=0.30, seed=42):
        files = sorted(files); rng = random.Random(seed)
        sh = files.copy(); rng.shuffle(sh)
        n = max(1, int(len(sh) * r)); return sh[n:], sh[:n]

    # ---------------------------------------------------------------------------
    # 3. MUSAN split
    # ---------------------------------------------------------------------------
    print("\nMUSAN split:")
    noise_test_banks = {}
    for i, (nt, sub) in enumerate(NOISE_SUBDIRS):
        files = list_audio(os.path.join(MUSAN_ROOT, sub))
        tr, te = split_strat(files, MUSAN_TEST_RATIO, MUSAN_SPLIT_SEED_BASE + i)
        assert not (set(tr) & set(te))
        noise_test_banks[nt] = te
        print(f"  {nt:8s}: test={len(te):5d}")

    # ---------------------------------------------------------------------------
    # 4. VOXCELEB1 TEST TRIALS
    # ---------------------------------------------------------------------------
    print("\n" + "=" * 78 + "\nLOADING VOXCELEB1 TEST TRIALS\n" + "=" * 78)
    if not os.path.isfile(TRIAL_FILE):
        urllib.request.urlretrieve(
            "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt", TRIAL_FILE)

    def load_trials(path):
        out = []
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    out.append((parts[1], parts[2], int(parts[0])))
        return out

    def resolve(rel):
        rel = rel.replace("\\", "/").lstrip("/")
        for base in (TEST_ROOT, os.path.join(TEST_ROOT, "wav"), "/marimo/voxceleb1_test"):
            p = os.path.join(base, rel)
            if os.path.isfile(p): return p
        mm = list(Path(TEST_ROOT).rglob(os.path.basename(rel)))
        return str(mm[0]) if mm else None

    all_trials = load_trials(TRIAL_FILE)
    resolved = []
    for e, t, lab in all_trials:
        ep, tp = resolve(e), resolve(t)
        if ep and tp:
            resolved.append((ep, tp, lab))
    assert not [(e, t) for e, t, _ in resolved if e == t]
    ue = sorted({e for e, _, _ in resolved})
    ut = sorted({t for _, t, _ in resolved})
    print(f"Resolved: {len(resolved)}  Enroll: {len(ue)}  Test: {len(ut)}")

    # ===========================================================================
    # 5. Hardcoded previous-run results (ReDimNet-B6, TitaNet-Large)
    # ===========================================================================
    def make_precomputed_df(name, clean_eer, per_noise_dicts):
        """Build a DataFrame with columns [noise_type, snr, eer, condition]."""
        rows = [{"noise_type": "clean", "snr": None, "eer": clean_eer, "condition": "clean"}]
        for nt, snr_map in per_noise_dicts.items():
            for snr, eer in snr_map.items():
                rows.append({"noise_type": nt, "snr": snr, "eer": eer, "condition": nt})
        return pd.DataFrame(rows)

    def report_precomputed(name, df):
        print("\n" + "=" * 80)
        print(f"{name} — EER (%) [from previous run, NOT re-evaluated]")
        print("=" * 80)
        p = df.pivot_table(index="noise_type", columns="snr", values="eer", dropna=False)
        p = p.reindex(["clean"] + [c for c in p.index if c != "clean"])
        print(p.round(2))

        nd = df[df["noise_type"] != "clean"]
        print(f"\nSummary by noise type ({name}):")
        for nt in ["babble", "music", "noise"]:
            s = nd[nd["noise_type"] == nt]
            if len(s): print(f"  {nt:<8} {s['eer'].mean():.2f}%")
        print(f"  {'ALL':<8} {nd['eer'].mean():.2f}%")
        pp = nd[nd["snr"].isin(PAPER_SNRS)]
        print(f"  Paper subset (0 to -20 dB): {pp['eer'].mean():.2f}%")
        cl = df[df["noise_type"] == "clean"].iloc[0]
        print(f"  Clean: {cl['eer']:.2f}%")
        return df

    redimnet_df = make_precomputed_df(
        "ReDimNet-B6",
        clean_eer=6.46,
        per_noise_dicts={
            "babble": {-20: 46.67, -15: 44.55, -10: 41.03, -5: 31.11, 0: 21.02,
                       5: 16.17, 10: 12.99, 15: 11.20, 20: 9.94},
            "music":  {-20: 41.54, -15: 38.50, -10: 32.43, -5: 26.88, 0: 21.37,
                       5: 16.82, 10: 13.29, 15: 11.06, 20: 9.77},
            "noise":  {-20: 32.57, -15: 26.97, -10: 22.82, -5: 18.66, 0: 15.44,
                       5: 12.92, 10: 11.56, 15: 10.40, 20: 9.73},
        },
    )
    report_precomputed("ReDimNet-B6", redimnet_df)

    titanet_df = make_precomputed_df(
        "TitaNet-Large",
        clean_eer=0.82,
        per_noise_dicts={
            "babble": {-20: 46.40, -15: 43.29, -10: 35.19, -5: 22.28, 0: 9.84,
                       5: 3.75, 10: 1.56, 15: 1.03, 20: 0.90},
            "music":  {-20: 40.68, -15: 33.18, -10: 21.89, -5: 8.86, 0: 3.19,
                       5: 1.45, 10: 1.11, 15: 0.95, 20: 0.89},
            "noise":  {-20: 25.00, -15: 16.10, -10: 8.30, -5: 3.15, 0: 1.82,
                       5: 1.31, 10: 1.05, 15: 0.96, 20: 0.91},
        },
    )
    report_precomputed("TitaNet-Large", titanet_df)

    # ===========================================================================
    # 6. MODEL LOADERS
    # ===========================================================================
    def load_wavlm_sv():
        from transformers import WavLMForXVector
        MODEL_ID = "microsoft/wavlm-base-plus-sv"
        print(f"\nLoading {MODEL_ID} ...")
        model = WavLMForXVector.from_pretrained(MODEL_ID).eval().to(DEVICE)
        for p in model.parameters(): p.requires_grad_(False)

        @torch.no_grad()
        def embed(wav, wav_len=None):
            B, T = wav.shape
            if wav_len is None:
                mask = torch.ones(B, T, dtype=torch.long, device=wav.device)
            else:
                abs_len = (wav_len.float() * T).round().long().clamp(min=1, max=T)
                arange = torch.arange(T, device=wav.device).unsqueeze(0).expand(B, T)
                mask = (arange < abs_len.unsqueeze(1)).long()
            out = model(input_values=wav, attention_mask=mask)
            return F.normalize(out.embeddings.float(), dim=-1)

        return embed, "WavLM-SV"

    def load_w2v_bert_2():
        from transformers import Wav2Vec2BertModel, SeamlessM4TFeatureExtractor
        MODEL_ID = "facebook/w2v-bert-2.0"
        print(f"\nLoading {MODEL_ID} ...")
        fe = SeamlessM4TFeatureExtractor.from_pretrained(MODEL_ID)
        model = Wav2Vec2BertModel.from_pretrained(MODEL_ID).eval().to(DEVICE)
        for p in model.parameters(): p.requires_grad_(False)

        @torch.no_grad()
        def embed(wav, wav_len=None):
            B, T = wav.shape
            if wav_len is not None:
                abs_len = (wav_len.float() * T).round().long().clamp(min=1, max=T)
                mask = torch.zeros_like(wav)
                for b in range(B):
                    mask[b, :abs_len[b]] = 1.0
                wav = wav * mask
            wav_np = wav.detach().cpu().numpy()
            inputs = fe(list(wav_np), sampling_rate=SR, return_tensors="pt", padding=True)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            out = model(**inputs)
            h = out.last_hidden_state
            emb = h.mean(dim=1)
            return F.normalize(emb.float(), dim=-1)

        return embed, "w2v-BERT-2.0"

    # ===========================================================================
    # 7. EMBEDDING HELPERS
    # ===========================================================================
    @torch.inference_mode()
    def embed_clean(embed_fn, paths, batch_size=8):
        cache = {}
        for i in range(0, len(paths), batch_size):
            batch = paths[i:i + batch_size]
            wavs = [load_wav(p, SR) for p in batch]
            L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
            ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
            pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)
            emb = embed_fn(pad, wl).cpu()
            for j, p in enumerate(batch):
                cache[p] = emb[j]
        return cache

    @torch.inference_mode()
    def embed_noisy(embed_fn, paths, nt=None, snr=None, noise_files=None,
                    batch_size=8, tag="test"):
        cache = {}
        for i in range(0, len(paths), batch_size):
            batch = paths[i:i + batch_size]
            wavs = [load_wav(p, SR) for p in batch]
            if snr is not None:
                mixed = []
                for p, w in zip(batch, wavs):
                    r = random.Random(sseed(tag, nt, snr, p))
                    npx = r.choice(noise_files); nz = load_wav(npx, SR)
                    no, _, _ = mix_det(w, nz, snr, (tag, nt, snr, p))
                    mixed.append(no)
                wavs = mixed
            L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
            ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
            pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)
            emb = embed_fn(pad, wl).cpu()
            for j, p in enumerate(batch):
                cache[p] = emb[j]
            del pad
        if USE_CUDA: torch.cuda.empty_cache()
        return cache

    # ===========================================================================
    # 8. SWEEP FOR ONE MODEL
    # ===========================================================================
    def run_sweep(embed_fn, model_name):
        print("\n" + "=" * 80)
        print(f"EVALUATING: {model_name}")
        print("=" * 80)

        print("Caching clean enrollment embeddings ...")
        enr_cache = embed_clean(embed_fn, ue, batch_size=8)
        print(f"  {len(enr_cache)} cached")

        conditions = [("clean", None, None)]
        for nt, _ in NOISE_SUBDIRS:
            for snr in EVAL_SNRS:
                conditions.append((nt, nt, snr))

        print(f"Evaluating {len(conditions)} conditions ...")
        results = []
        t0 = time.time()

        for cond, nt, snr in tqdm(conditions, desc=model_name):
            nf = noise_test_banks[nt] if snr is not None else None
            tst_cache = embed_noisy(embed_fn, ut, nt=nt, snr=snr,
                                    noise_files=nf, batch_size=8)

            scores, labels = [], []
            for e, t, l in resolved:
                ee = enr_cache[e].to(DEVICE)
                te = tst_cache[t].unsqueeze(0).to(DEVICE)
                scores.append(F.cosine_similarity(ee.unsqueeze(0), te).item())
                labels.append(l)

            eer = compute_eer(np.array(scores), np.array(labels))
            results.append({
                "condition": cond,
                "noise_type": nt if snr is not None else "clean",
                "snr": snr,
                "eer": eer,
            })
            gc.collect()
            if USE_CUDA: torch.cuda.empty_cache()

        print(f"{model_name} sweep time: {(time.time() - t0) / 60:.1f} min")
        df = pd.DataFrame(results)
        df.to_csv(f"/marimo/{model_name.replace('-', '_').lower()}_voxceleb1_musan.csv", index=False)

        def piv(c):
            p = df.pivot_table(index="noise_type", columns="snr", values=c, dropna=False)
            return p.reindex(["clean"] + [c for c in p.index if c != "clean"])

        print(f"\nEER (%) — {model_name}")
        print(piv("eer").round(2))

        nd = df[df["condition"] != "clean"]
        print(f"\nSummary by noise type ({model_name}):")
        for nt in ["babble", "music", "noise"]:
            s = nd[nd["noise_type"] == nt]
            print(f"  {nt:<8} {s['eer'].mean():.2f}%")
        print(f"  {'ALL':<8} {nd['eer'].mean():.2f}%")
        pp = nd[nd["snr"].isin(PAPER_SNRS)]
        print(f"  Paper subset (0 to -20 dB): {pp['eer'].mean():.2f}%")
        cl = df[df["condition"] == "clean"].iloc[0]
        print(f"  Clean: {cl['eer']:.2f}%")

        return df

    # ===========================================================================
    # 9. RUN REMAINING MODELS
    # ===========================================================================
    dfs = {
        "ReDimNet-B6": redimnet_df,
        "TitaNet-Large": titanet_df,
    }

    embed_fn, name = load_wavlm_sv()
    dfs[name] = run_sweep(embed_fn, name)
    del embed_fn; gc.collect()
    if USE_CUDA: torch.cuda.empty_cache()

    embed_fn, name = load_w2v_bert_2()
    dfs[name] = run_sweep(embed_fn, name)
    del embed_fn; gc.collect()
    if USE_CUDA: torch.cuda.empty_cache()

    # ===========================================================================
    # 10. FINAL COMPARISON (filter on noise_type, not condition)
    # ===========================================================================
    print("\n" + "=" * 80)
    print("FINAL COMPARISON — mean EER across all noisy conditions")
    print("=" * 80)
    print(f"{'Model':<20} {'Clean':>8} {'All noisy':>10} {'Paper (-20..0)':>15}")
    print("-" * 55)
    rows = []
    for name, df in dfs.items():
        nd = df[df["noise_type"] != "clean"]
        pp = nd[nd["snr"].isin(PAPER_SNRS)]
        cl_row = df[df["noise_type"] == "clean"]
        clean_eer = cl_row["eer"].iloc[0] if len(cl_row) else float("nan")
        all_noisy = nd["eer"].mean()
        paper = pp["eer"].mean()
        rows.append({
            "Model": name,
            "Clean": clean_eer,
            "All noisy": all_noisy,
            "Paper (-20..0)": paper,
        })
        print(f"{name:<20} {clean_eer:>8.2f} {all_noisy:>10.2f} {paper:>15.2f}")

    comparison_df = pd.DataFrame(rows)
    comparison_df.to_csv("/marimo/final_comparison.csv", index=False)

    # Markdown output
    print("\n" + "=" * 80)
    print("MARKDOWN: FINAL COMPARISON")
    print("=" * 80)
    print(comparison_df.to_markdown(index=False, floatfmt=".2f"))

    return dfs, comparison_df


dfs, comparison_df = compare()

## Results

### ReDimNet-B6 — EER (%)

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---|---|---|---|---|---|---|---|---|---|
| babble | 46.67 | 44.55 | 41.03 | 31.11 | 21.02 | 16.17 | 12.99 | 11.20 | 9.94 | — |
| music | 41.54 | 38.50 | 32.43 | 26.88 | 21.37 | 16.82 | 13.29 | 11.06 | 9.77 | — |
| noise | 32.57 | 26.97 | 22.82 | 18.66 | 15.44 | 12.92 | 11.56 | 10.40 | 9.73 | — |
| clean | — | — | — | — | — | — | — | — | — | 6.46 |


| Noise type | EER % |
|---|---|
| babble | 26.08 |
| music | 23.52 |
| noise | 17.90 |
| **ALL** | **22.50** |
| Paper subset (0 to −20 dB) | 30.54 |
| Clean | 6.46 |

### TitaNet-Large — EER (%)

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---|---|---|---|---|---|---|---|---|---|
| babble | 46.40 | 43.29 | 35.19 | 22.28 | 9.84 | 3.75 | 1.56 | 1.03 | 0.90 | — |
| music | 40.68 | 33.18 | 21.89 | 8.86 | 3.19 | 1.45 | 1.11 | 0.95 | 0.89 | — |
| noise | 25.00 | 16.10 | 8.30 | 3.15 | 1.82 | 1.31 | 1.05 | 0.96 | 0.91 | — |
| clean | — | — | — | — | — | — | — | — | — | 0.82 |

| Noise type | EER % |
|---|---|
| babble | 18.25 |
| music | 12.47 |
| noise | 6.51 |
| **ALL** | **12.41** |
| Paper subset (0 to −20 dB) | 21.28 |
| Clean | 0.82 |

### WavLM-SV — EER (%)

| noise_type | -20.0 | -10.0 | 0.0 | 10.0 | 20.0 | clean |
|---|---|---|---|---|---|---|
| babble | 47.17 | 38.61 | 18.38 | 8.13 | 5.95 | — |
| music | 44.73 | 35.86 | 18.10 | 7.63 | 5.99 | — |
| noise | 37.18 | 24.80 | 13.52 | 7.68 | 6.06 | — |
| clean | — | — | — | — | — | 5.47 |

| Noise type | EER % |
|---|---|
| babble | 23.65 |
| music | 22.46 |
| noise | 17.85 |
| **ALL** | **21.32** |
| Paper subset (0 to −20 dB) | 30.93 |
| Clean | 5.47 |

### w2v-BERT-2.0 — EER (%)

| noise_type | -20.0 | -10.0 | 0.0 | 10.0 | 20.0 | clean |
|---|---|---|---|---|---|---|
| babble | 48.75 | 48.19 | 46.84 | 46.14 | 45.55 | — |
| music | 49.16 | 48.70 | 47.89 | 46.90 | 45.83 | — |
| noise | 49.10 | 48.43 | 47.16 | 46.35 | 45.41 | — |
| clean | — | — | — | — | — | 44.93 |

| Noise type | EER % |
|---|---|
| babble | 47.09 |
| music | 47.69 |
| noise | 47.29 |
| **ALL** | **47.36** |
| Paper subset (0 to −20 dB) | 48.25 |
| Clean | 44.93 |

### Final comparison

| Model | Clean | All noisy | Paper (−20..0) |
|---|---|---|---|
| ReDimNet-B6 | 6.46 | 22.50 | 30.54 |
| **TitaNet-Large** | **0.82** | **12.41** | **21.28** |
| WavLM-SV | 5.47 | 21.32 | 30.93 |
| w2v-BERT-2.0 | 44.93 | 47.36 | 48.25 |
